# Goal 50 Full 100M Verified-Code Quality Comparison on Colab L4

This notebook runs the full 100M comparison only after the Goal 50 memorization gate passes. It compares Dense, MoP Full, Warm Adapter/Norm/Head 128, Cached Adapter/Norm/Head 128, and Cached Tail-Only LoRA Rank 8 on one balanced 10,000-lesson verified code-repair dataset.

Every enabled profile uses the same fixed split, tokenizer, sequence length, batch policy, optimizer-update budget, full held-out loss evaluation, and stratified generated-code evaluation from its best checkpoint. No quantization is used. Checkpoints, optimizer state, and activation caches stay outside the lightweight report.


## 0. Settings

First run `colab_l4_goal50_100m_learning_gate.ipynb`. Keep its passing `learning_gate.json` available in the same Colab session or upload it when prompted below.

Set `TARGET_EVAL_LOSS` from a completed baseline calibration before training. The notebook refuses to run without this shared target because baseline time-to-target cannot be reconstructed afterward.


In [ ]:
REPO_URL = "https://github.com/NikanBHMNJ/MoP.git"
REPO_DIR = "/content/MoP"
REPORT_ID = "goal50_100m_quality_comparison"
REPORT_DIR = f"reports/{REPORT_ID}"
GATE_REPORT_PATH = "reports/goal50_100m_learning_gate/learning_gate.json"
REQUIRE_MEMORIZATION_GATE = True

OPTIMIZER_UPDATES = 2000
GRADIENT_ACCUMULATION_STEPS = 8
MAX_STEPS = OPTIMIZER_UPDATES * GRADIENT_ACCUMULATION_STEPS
EVAL_EVERY_OPTIMIZER_UPDATES = 100
EVAL_EVERY_STEPS = EVAL_EVERY_OPTIMIZER_UPDATES * GRADIENT_ACCUMULATION_STEPS
SAVE_EVERY_OPTIMIZER_UPDATES = 500
SAVE_EVERY_STEPS = SAVE_EVERY_OPTIMIZER_UPDATES * GRADIENT_ACCUMULATION_STEPS
MAX_TRAIN_EXAMPLES = 10000
MAX_EVAL_EXAMPLES = 1000
COUNT_PER_CATEGORY = 2000
SPLIT_SEED = 42
TRAIN_SHUFFLE_SEED = 42
QUALITY_FORMAT = "fixed_code_xml"
ADAPTER_BOTTLENECK = 128

# Required: set this from a prior Dense/MoP Full baseline calibration.
TARGET_EVAL_LOSS = None
TARGET_EVAL_LOSS_WAS_CONFIGURED = TARGET_EVAL_LOSS is not None

CACHE_TEACHER_TOP_K = 16
CACHE_RECORDS_PER_SHARD = 256
DISTILLATION_WEIGHT = 0.2
DISTILLATION_TEMPERATURE = 2.0
HARD_EXAMPLE_REPLAY = True
HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD = 1.5
HARD_EXAMPLE_REPLAY_MULTIPLIER = 2

# A later-run subset is allowed, but it is deterministically balanced across all five bug types.
GENERATION_EVAL_EXAMPLES = 250
GENERATION_MAX_NEW_TOKENS = 256

RUN_DENSE = True
RUN_MOP_FULL = True
RUN_WARM_ADAPTER_128 = True
RUN_CACHED_ADAPTER_128 = True
RUN_CACHED_LORA_8 = True
RUN_CACHED_LORA_16 = False
TEACHER_LABEL = "mop_full"
REQUIRE_CUDA = True


In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import re
import shutil
import subprocess
from pathlib import Path


def sh(command: str, *, check: bool = True) -> str:
    print(f"\n$ {command}")
    result = subprocess.run(
        command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {command}")
    return result.stdout


def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path: str | Path, data) -> Path:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return output


def parse_key(output: str, key: str) -> str:
    match = re.search(rf"^{re.escape(key)}=(.+)$", output, flags=re.MULTILINE)
    if not match:
        raise ValueError(f"missing {key}=... in command output")
    return match.group(1).strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 1. Clone, Install, And Verify The L4

In [ ]:
repo_dir = Path(REPO_DIR)
if not (repo_dir / ".git").exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    sh(f"git clone --depth 1 {REPO_URL} {repo_dir}")
else:
    sh(f"git -C {repo_dir} pull --ff-only", check=False)
os.chdir(repo_dir)

sh("python -m pip install -q -e .[dev,gpu]")
sh("mopforge version")
sh("mopforge doctor")
sh("mopforge runtime detect")
sh("nvidia-smi", check=False)

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this report notebook.")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

gate_path = Path(GATE_REPORT_PATH)
if REQUIRE_MEMORIZATION_GATE and not gate_path.exists():
    try:
        from google.colab import files

        print("Upload learning_gate.json from the Goal 50 memorization notebook.")
        uploaded = files.upload()
        if "learning_gate.json" in uploaded:
            gate_path = Path("learning_gate.json")
    except ImportError:
        pass
if REQUIRE_MEMORIZATION_GATE:
    if not gate_path.exists():
        raise FileNotFoundError("A passing Goal 50 learning_gate.json is required.")
    memorization_gate = read_json(gate_path)
    if memorization_gate.get("passed") is not True:
        raise RuntimeError("The 100M memorization gate did not pass. Do not run this comparison.")
    print("Memorization gate: PASS", gate_path)

if TARGET_EVAL_LOSS is None:
    raise ValueError(
        "Set TARGET_EVAL_LOSS from a completed baseline calibration before training."
    )


## 2. Build One Balanced 10,000-Lesson Verified Split


In [ ]:
dataset_output = sh(
    " ".join(
        [
            "mopforge gpu prepare-efficiency-data",
            f"--dataset-id {REPORT_ID}",
            f"--count-per-category {COUNT_PER_CATEGORY}",
            f"--split-seed {SPLIT_SEED}",
            "--stratify-by bug_type",
            f"--quality-format {QUALITY_FORMAT}",
            "--verify",
            "--overwrite",
        ]
    )
)
DATASET_REF = parse_key(dataset_output, "dataset_ref")
DATASET_SPLIT_ID = parse_key(dataset_output, "split_id")
print("DATASET_REF=", DATASET_REF)
print("DATASET_SPLIT_ID=", DATASET_SPLIT_ID)


## 3. Create Comparable L4 Configs

In [ ]:
CONFIG_DIR = Path("configs/jobs/colab_l4_goal50_100m_quality")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNS: dict[str, dict] = {}

COMMON_OVERRIDES = {
    "device": "cuda",
    "precision": "bf16",
    "require_device_available": True,
    "dataset_ref": DATASET_REF,
    "dataset_split_id": DATASET_SPLIT_ID,
    "lesson_path": "data/coding_bugfix_efficiency_lessons.jsonl",
    "max_steps": int(MAX_STEPS),
    "gradient_accumulation_steps": int(GRADIENT_ACCUMULATION_STEPS),
    "eval_full_dataset": True,
    "shuffle_train": True,
    "train_shuffle_seed": int(TRAIN_SHUFFLE_SEED),
    "eval_every_steps": int(EVAL_EVERY_STEPS),
    "save_every_steps": int(SAVE_EVERY_STEPS),
    "max_train_examples": int(MAX_TRAIN_EXAMPLES),
    "max_eval_examples": int(MAX_EVAL_EXAMPLES),
    "run_generation_eval": True,
    "generation_eval_examples": int(GENERATION_EVAL_EXAMPLES),
    "generation_max_new_tokens": int(GENERATION_MAX_NEW_TOKENS),
    "generation_eval_use_best_checkpoint": True,
    "generation_eval_include_train": False,
    "generation_eval_stratify_by": "bug_type",
    "output_root": "gpu_runs",
    "artifact_root": "artifacts",
}


def make_config(label: str, template: str, overrides: dict | None = None, *, include_target: bool = True) -> Path:
    envelope = read_json(template)
    payload = dict(envelope["payload"])
    payload.update(COMMON_OVERRIDES)
    payload.update(overrides or {})
    if include_target and TARGET_EVAL_LOSS is not None:
        payload["target_eval_loss"] = float(TARGET_EVAL_LOSS)
    else:
        payload.pop("target_eval_loss", None)
    payload["name"] = f"{REPORT_ID}_{label}"
    metadata = dict(payload.get("metadata") or {})
    metadata.update(
        {
            "report_id": REPORT_ID,
            "label": label,
            "dataset_ref": DATASET_REF,
            "dataset_split_id": DATASET_SPLIT_ID,
            "split_seed": SPLIT_SEED,
            "train_shuffle_seed": TRAIN_SHUFFLE_SEED,
            "optimizer_updates_requested": OPTIMIZER_UPDATES,
            "microsteps_requested": MAX_STEPS,
            "eval_full_dataset": True,
            "quality_format": QUALITY_FORMAT,
            "hardware_target": "google_colab_l4",
            "no_quantization": True,
        }
    )
    payload["metadata"] = metadata
    output = CONFIG_DIR / f"{label}.json"
    return write_json(
        output,
        {
            "kind": "gpu_train",
            "version": envelope.get("version", "1"),
            "metadata": envelope.get("metadata", {}),
            "payload": payload,
        },
    )


def train_config(label: str, config_path: Path) -> dict:
    sh(f"mopforge gpu validate {config_path}")
    output = sh(f"mopforge gpu train {config_path}")
    run_id = parse_key(output, "run_id")
    latest_checkpoint = parse_key(output, "latest_checkpoint_path")
    result = read_json(Path("gpu_runs") / run_id / "gpu_training_result.json")
    checkpoint_path = (
        result.get("artifacts", {}).get("best_checkpoint_path")
        or result.get("state", {}).get("best_checkpoint_path")
        or latest_checkpoint
    )
    record = {
        "run_id": run_id,
        "checkpoint_path": checkpoint_path,
        "config_path": str(config_path),
    }
    RUNS[label] = record
    return record


dense_config = make_config(
    "dense", "configs/jobs/100m_dense_extended_efficiency.json", include_target=True
)
mop_full_config = make_config(
    "mop_full",
    "configs/jobs/100m_mop_full_extended_efficiency.json",
    {"fast_adapter_bottleneck_dim": ADAPTER_BOTTLENECK},
    include_target=True,
)

## 4. Train Dense, MoP Full, And Warm Adapter 128


In [ ]:
if RUN_DENSE:
    train_config("dense", dense_config)
if RUN_MOP_FULL:
    train_config("mop_full", mop_full_config)

if RUN_WARM_ADAPTER_128:
    if "mop_full" not in RUNS:
        raise RuntimeError("Warm Adapter 128 requires RUN_MOP_FULL=True.")
    warm_config = make_config(
        "warm_adapter_norm_head_128",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        {
            "fast_adapter_bottleneck_dim": ADAPTER_BOTTLENECK,
            "resume_from_checkpoint": RUNS["mop_full"]["checkpoint_path"],
            "base_checkpoint_path": RUNS["mop_full"]["checkpoint_path"],
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
        },
    )
    train_config("warm_adapter_norm_head_128", warm_config)


## 5. Cache Teacher Signals And Train Sparse Students

The cache stores bf16 boundary states, top-k teacher logits, and teacher CE loss. Tail-only LoRA keeps its low-rank deltas after the frozen cache boundary, allowing the backbone to be offloaded during training.

In [ ]:
if TEACHER_LABEL not in RUNS:
    raise RuntimeError(f"Teacher run {TEACHER_LABEL!r} is unavailable.")
if "warm_adapter_norm_head_128" not in RUNS:
    raise RuntimeError("Cached students require the Warm Adapter 128 checkpoint.")

teacher_checkpoint = RUNS[TEACHER_LABEL]["checkpoint_path"]
warm_checkpoint = RUNS["warm_adapter_norm_head_128"]["checkpoint_path"]
cache_source_config = make_config(
    "teacher_cache_source",
    "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
    {
        "fast_adapter_bottleneck_dim": ADAPTER_BOTTLENECK,
        "resume_from_checkpoint": teacher_checkpoint,
        "base_checkpoint_path": teacher_checkpoint,
        "resume_model_only": True,
        "save_full_checkpoints": False,
        "run_generation_eval": False,
    },
)
cache_path = Path("outputs") / f"{REPORT_ID}_teacher_topk_cache_manifest.json"
sh(
    " ".join(
        [
            "mopforge gpu cache-activations",
            str(cache_source_config),
            f"--checkpoint {teacher_checkpoint}",
            f"--output {cache_path}",
            "--dtype bf16",
            f"--teacher-top-k {CACHE_TEACHER_TOP_K}",
            f"--records-per-shard {CACHE_RECORDS_PER_SHARD}",
        ]
    )
)

CACHED_COMMON = {
    "fast_adapter_bottleneck_dim": ADAPTER_BOTTLENECK,
    "resume_from_checkpoint": warm_checkpoint,
    "base_checkpoint_path": warm_checkpoint,
    "resume_model_only": True,
    "save_trainable_only_checkpoints": True,
    "activation_cache_path": str(cache_path),
    "offload_frozen_backbone_for_cache": True,
    "distillation_enabled": True,
    "distillation_weight": float(DISTILLATION_WEIGHT),
    "distillation_temperature": float(DISTILLATION_TEMPERATURE),
    "distillation_top_k": int(CACHE_TEACHER_TOP_K),
    "hard_example_replay_enabled": bool(HARD_EXAMPLE_REPLAY),
    "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
    "hard_example_replay_multiplier": int(HARD_EXAMPLE_REPLAY_MULTIPLIER),
}

if RUN_CACHED_ADAPTER_128:
    config = make_config(
        "cached_warm_adapter_norm_head_128",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        CACHED_COMMON,
    )
    train_config("cached_warm_adapter_norm_head_128", config)

for rank, enabled in ((8, RUN_CACHED_LORA_8), (16, RUN_CACHED_LORA_16)):
    if not enabled:
        continue
    label = f"cached_warm_lora_rank{rank}"
    config = make_config(
        label,
        "configs/jobs/100m_mop_warm_lora_deltas_efficiency.json",
        {
            **CACHED_COMMON,
            "use_lora_deltas": True,
            "lora_tail_only": True,
            "lora_rank": rank,
        },
    )
    train_config(label, config)

## 6. Build The Lightweight Report

In [ ]:
REPORT_PATH = Path(REPORT_DIR)
if REPORT_PATH.exists():
    shutil.rmtree(REPORT_PATH)
REPORT_PATH.mkdir(parents=True, exist_ok=True)

LIGHTWEIGHT_FILES = [
    "config.json",
    "gpu_training_result.json",
    "metrics.json",
    "runtime.json",
    "state.json",
    "memory_estimate.json",
    "generation_eval.json",
    "ground_truth_controls.json",
]
for label, record in RUNS.items():
    source = Path("gpu_runs") / record["run_id"]
    destination = REPORT_PATH / "runs" / label
    destination.mkdir(parents=True, exist_ok=True)
    for filename in LIGHTWEIGHT_FILES:
        candidate = source / filename
        if candidate.exists():
            shutil.copy2(candidate, destination / filename)

write_json(REPORT_PATH / "run_manifest.json", RUNS)
write_json(
    REPORT_PATH / "experiment_settings.json",
    {
        "report_id": REPORT_ID,
        "created_utc": dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
        "dataset_ref": DATASET_REF,
        "dataset_split_id": DATASET_SPLIT_ID,
        "split_seed": SPLIT_SEED,
        "quality_format": QUALITY_FORMAT,
        "target_eval_loss": TARGET_EVAL_LOSS,
        "target_was_auto_derived": False,
        "optimizer_updates": OPTIMIZER_UPDATES,
        "microsteps": MAX_STEPS,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "eval_full_dataset": True,
        "split_stratify_by": "bug_type",
        "train_shuffle_seed": TRAIN_SHUFFLE_SEED,
        "generation_eval_stratify_by": "bug_type",
        "adapter_bottleneck": ADAPTER_BOTTLENECK,
        "cache_teacher_top_k": CACHE_TEACHER_TOP_K,
        "distillation_weight": DISTILLATION_WEIGHT,
        "distillation_temperature": DISTILLATION_TEMPERATURE,
        "hard_example_replay": HARD_EXAMPLE_REPLAY,
        "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
        "generation_eval_examples": GENERATION_EVAL_EXAMPLES,
        "hardware_target": "google_colab_l4",
        "quantization": None,
    },
)

run_ids = [record["run_id"] for record in RUNS.values()]
sh(
    " ".join(
        [
            "mopforge gpu compare-runs",
            *run_ids,
            f"--output {REPORT_PATH / 'comparison.json'}",
            f"--output-csv {REPORT_PATH / 'comparison.csv'}",
        ]
    )
)

In [ ]:
rows = read_json(REPORT_PATH / "comparison.json").get("runs", [])
label_by_run_id = {record["run_id"]: label for label, record in RUNS.items()}


def cell(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


table = [
    "| Profile | Train loss | Eval loss | Best eval | Target sec | Tokens/sec | Peak alloc GB | Peak reserved GB | Final reserved GB | Trainable ratio | Checkpoint MB | Exact match | Syntax pass | Verifier pass |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in rows:
    table.append(
        "| "
        + " | ".join(
            [
                label_by_run_id.get(row.get("run_id"), row.get("run_id", "")),
                cell(row.get("train_loss")),
                cell(row.get("eval_loss")),
                cell(row.get("best_eval_loss")),
                cell(row.get("time_to_target_loss_sec")),
                cell(row.get("tokens_per_sec")),
                cell(row.get("peak_allocated_gb")),
                cell(row.get("peak_reserved_gb")),
                cell(row.get("final_reserved_gb")),
                cell(row.get("trainable_param_ratio")),
                cell(row.get("checkpoint_size_mb")),
                cell(row.get("gen_exact_match_rate")),
                cell(row.get("gen_syntax_pass_rate")),
                cell(row.get("gen_verifier_pass_rate")),
            ]
        )
        + " |"
    )

evidence = {}
row_by_label = {
    label_by_run_id.get(row.get("run_id"), row.get("run_id", "")): row for row in rows
}
for label, record in RUNS.items():
    run_dir = Path("gpu_runs") / record["run_id"]
    metrics = read_json(run_dir / "metrics.json")
    generation = read_json(run_dir / "generation_eval.json")
    controls = read_json(run_dir / "ground_truth_controls.json")
    summary = generation["splits"]["eval"]["summary"]
    evidence[label] = {
        "ground_truth_controls_passed": controls["passed"],
        "generation_checkpoint": generation["checkpoint"],
        "category_count": len(summary.get("per_category", {})),
        "fixed_code_complete_rate": summary["gen_fixed_code_complete_rate"],
        "syntax_pass_rate": summary["gen_syntax_pass_rate"],
        "verifier_pass_rate": summary["gen_verifier_pass_rate"],
        "exact_match_rate": summary["gen_exact_match_rate"],
        "latest_eval_examples": metrics.get("latest_eval_examples"),
        "full_eval_examples": metrics.get("data", {}).get("eval_examples"),
        "optimizer_updates": metrics.get("optimizer_steps"),
        "sequence_length_statistics": metrics.get("data", {}).get("sequence_length_statistics"),
    }

dense_row = row_by_label.get("dense", {})
cached_labels = [label for label in RUNS if label.startswith("cached_")]


def numeric(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def untruncated(item):
    stats = item.get("sequence_length_statistics") or {}
    return all(
        (stats.get(split_name) or {}).get("truncated_examples") == 0
        for split_name in ("train", "eval")
    )


def generation_budget_covers_targets(item):
    stats = item.get("sequence_length_statistics") or {}
    maxima = [
        (stats.get(split_name) or {}).get("max_target_tokens")
        for split_name in ("train", "eval")
    ]
    return all(numeric(value) and value <= GENERATION_MAX_NEW_TOKENS for value in maxima)


checks = {
    "all_ground_truth_controls": all(item["ground_truth_controls_passed"] for item in evidence.values()),
    "all_generation_from_best_checkpoint": all(item["generation_checkpoint"]["source"] == "best_eval" for item in evidence.values()),
    "all_five_categories": all(item["category_count"] == 5 for item in evidence.values()),
    "full_held_out_loss": all(item["latest_eval_examples"] == item["full_eval_examples"] for item in evidence.values()),
    "optimizer_update_budget": all(item["optimizer_updates"] == OPTIMIZER_UPDATES for item in evidence.values()),
    "no_train_or_eval_truncation": all(untruncated(item) for item in evidence.values()),
    "generation_budget_covers_targets": all(generation_budget_covers_targets(item) for item in evidence.values()),
    "cached_complete_framing_at_least_90pct": all(evidence[label]["fixed_code_complete_rate"] >= 0.90 for label in cached_labels),
    "cached_syntax_at_least_80pct": all(evidence[label]["syntax_pass_rate"] >= 0.80 for label in cached_labels),
    "cached_verifier_at_least_20pct": all(evidence[label]["verifier_pass_rate"] >= 0.20 for label in cached_labels),
    "cached_exact_match_nonzero": all(evidence[label]["exact_match_rate"] > 0.0 for label in cached_labels),
    "cached_peak_reserved_below_dense": numeric(dense_row.get("peak_reserved_gb")) and all(numeric(row_by_label.get(label, {}).get("peak_reserved_gb")) and row_by_label[label]["peak_reserved_gb"] < dense_row["peak_reserved_gb"] for label in cached_labels),
    "cached_throughput_at_least_80pct_dense": numeric(dense_row.get("tokens_per_sec")) and all(numeric(row_by_label.get(label, {}).get("tokens_per_sec")) and row_by_label[label]["tokens_per_sec"] >= 0.8 * dense_row["tokens_per_sec"] for label in cached_labels),
}
acceptance = {
    "passed": bool(cached_labels) and all(checks.values()),
    "checks": checks,
    "thresholds": {
        "fixed_code_complete_rate": 0.90,
        "syntax_pass_rate": 0.80,
        "verifier_pass_rate": 0.20,
        "exact_match_rate": "greater_than_zero",
        "cached_tokens_per_sec_vs_dense": 0.80,
    },
    "evidence": evidence,
}
write_json(REPORT_PATH / "acceptance_gates.json", acceptance)

readme = f"""# Goal 50 Verified Code Quality L4 Report

Generated by `notebooks/colab_l4_goal50_100m_quality_comparison.ipynb`.

## Scope

- Dataset: `{DATASET_REF}`
- Split: `{DATASET_SPLIT_ID}` (seed `{SPLIT_SEED}`)
- Quality format: `{QUALITY_FORMAT}`
- Hardware: Google Colab L4
- Target eval loss: `{TARGET_EVAL_LOSS}`
- Teacher top-k: `{CACHE_TEACHER_TOP_K}`
- Hard-example replay: `{HARD_EXAMPLE_REPLAY}`
- Quantization: none
- Optimizer updates per profile: `{OPTIMIZER_UPDATES}`
- Full held-out loss evaluation: `True`
- Generation subset: `{GENERATION_EVAL_EXAMPLES}`, stratified across five bug types
- Acceptance gates: `{'PASS' if acceptance['passed'] else 'FAIL'}`

## Results

{chr(10).join(table)}

## Evidence

Each `runs/<profile>/generation_eval.json` records the exact best checkpoint, stratified generated samples, complete XML, syntax, exact-match, verifier, and per-category outcomes. `ground_truth_controls.json` must remain 100% passing. Cached-training VRAM is measured before the full model is temporarily restored for post-training quality evaluation. The report excludes activation caches, optimizer state, checkpoints, and model weights.

## Interpretation

`acceptance_gates.json` is the claim boundary. Do not scale to 1B or claim quality-preserving efficiency unless it passes; inspect the recorded per-category failures when it does not.
"""
(REPORT_PATH / "README.md").write_text(readme, encoding="utf-8")

forbidden_suffixes = {".pt", ".pth", ".ckpt", ".bin", ".safetensors"}
forbidden_files = [
    str(path.relative_to(REPORT_PATH))
    for path in REPORT_PATH.rglob("*")
    if path.is_file() and path.suffix.lower() in forbidden_suffixes
]
if forbidden_files:
    raise RuntimeError(f"model/checkpoint files entered report: {forbidden_files}")

manifest = {
    "report_id": REPORT_ID,
    "forbidden_model_files_detected": False,
    "files": [
        {
            "path": str(path.relative_to(REPORT_PATH)),
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in sorted(REPORT_PATH.rglob("*"))
        if path.is_file()
    ],
}
write_json(REPORT_PATH / "report_manifest.json", manifest)
print((REPORT_PATH / "README.md").read_text(encoding="utf-8"))

## 7. Download The Report

The ZIP contains only lightweight comparison evidence, acceptance gates, generated samples, controls, and metadata. It excludes model weights, optimizer state, and activation caches.


In [ ]:
zip_base = Path("reports") / REPORT_ID
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=REPORT_PATH.parent, base_dir=REPORT_PATH.name))
print("Report folder:", REPORT_PATH)
print("Report ZIP:", zip_path, f"({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")

try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab; the ZIP remains at", zip_path)